In [1]:
import pyrosetta
from benchmark import bpti_ryfyn_benchmark
from rotamer_extraction import extract_top_n_rotamers
from hamiltonian_creation import extract_hamiltonian_tensors, build_ising_hamiltonian, reduce_hamiltonian
from initialisation import initialize_rosetta

import numpy as np

initialize_rosetta(pyrosetta, extra_flags="-mute all")


Initializing PyRosetta with cleaning flags: -ignore_unrecognized_res and extra flags: -mute all
┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python313.m1 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [2]:
benchmark_pose = bpti_ryfyn_benchmark()

results = extract_top_n_rotamers(benchmark_pose)
pyrosetta.rosetta.core.io.pdb.dump_pdb(benchmark_pose, "benchmark_pose.pdb")

Fragment Sequence: RYFYN
Total Residues: 5
Creating score function
Pose scored successfully!
Creating Repacking Task - Core Rotamer Optimisation Protocol
Computing One-Body and Two-Body Energies
Iterating through molten residues - determining the top rotamer positions for each amino acid
1 1
2 2
3 3
4 4
5 5


True

In [3]:
rotamer_lib, ig, rot_sets = results
h_linear, J_quadratic = extract_hamiltonian_tensors(rotamer_lib, ig, rot_sets)


In [4]:
h_flex_linear, J_flex_quadratic, global_offset = reduce_hamiltonian(h_linear, J_quadratic, rotamer_lib)
for idx in h_linear:
    print(h_linear[idx])
    print(h_flex_linear.get(idx, "None"))
    print("\n---------------------------------------\n")

{0: 1.2480376958847046, 1: 1.5509175062179565, 2: 1.6199729442596436, 3: 1.8183133602142334}
{0: 1.2480376958847046, 1: 1.5509175062179565, 2: 1.6199729442596436, 3: 1.8183133602142334}

---------------------------------------

{0: 2.0819952487945557, 1: 2.081996440887451, 2: 3.0249295234680176, 3: 3.024930477142334}
{0: 2.0819952487945557, 1: 2.081996440887451, 2: 3.0249295234680176, 3: 3.024930477142334}

---------------------------------------

{0: 2.783350706100464, 1: 3.331935405731201, 2: 4.205162048339844}
{0: 2.783350706100464, 1: 3.331935405731201, 2: 4.205162048339844}

---------------------------------------

{0: 2.8021299839019775, 1: 2.802131175994873, 2: 3.082185983657837, 3: 3.0821871757507324}
{0: 2.8021299839019775, 1: 2.802131175994873, 2: 3.082185983657837, 3: 3.0821871757507324}

---------------------------------------

{0: 0.20771783590316772, 1: 0.33269232511520386, 2: 0.6052141785621643, 3: 1.415124535560608}
{0: 0.20771783590316772, 1: 0.33269232511520386, 2: 0.

In [5]:
hamiltonian = build_ising_hamiltonian(h_flex_linear, J_flex_quadratic, global_offset)

Reduced Hamiltonian built: 19 Qubits, 143 Pauli strings.


In [6]:
print(hamiltonian)

4884.384590045549 * I(0) + -508.58522180840373 * Z(0) + -507.591564707458 * Z(1) + -507.5886814855039 * Z(2) + -507.92053692787886 * Z(3) + -508.0840256139636 * Z(4) + -508.08402621001005 * Z(5) + -508.6215684339404 * Z(6) + -508.62156891077757 * Z(7) + -247.90659865830094 * Z(8) + -592.9867805019021 * Z(9) + -248.9889537319541 * Z(10) + -499.7665811404586 * Z(11) + -499.73405902832747 * Z(12) + -500.50922998040915 * Z(13) + -500.5092305764556 * Z(14) + -595.4202584847808 * Z(15) + -584.6821173764765 * Z(16) + -570.5111815556884 * Z(17) + -592.1065326826647 * Z(18) + 250.0 * (Z(0) @ Z(1)) + 250.0 * (Z(0) @ Z(2)) + 250.0 * (Z(0) @ Z(3)) + 2.239682912826538 * (Z(0) @ Z(4)) + 2.239682912826538 * (Z(0) @ Z(5)) + 2.2561678886413574 * (Z(0) @ Z(6)) + 2.2561678886413574 * (Z(0) @ Z(7)) + -0.5157089829444885 * (Z(0) @ Z(8)) + -0.04511748626828194 * (Z(0) @ Z(9)) + -0.46967217326164246 * (Z(0) @ Z(10)) + 250.0 * (Z(1) @ Z(2)) + 250.0 * (Z(1) @ Z(3)) + 1.870367407798767 * (Z(1) @ Z(4)) + 1.87036

In [7]:
import pennylane as qml
from pennylane import numpy as np

H_ising = hamiltonian
# Assuming 'H_ising' is the Hamiltonian returned by your build_ising_hamiltonian function
num_qubits = 19
dev = qml.device('default.qubit', wires=range(num_qubits))

# Define the standard X-mixer for QAOA
mixer_h = qml.dot([1.0] * num_qubits, [qml.PauliX(i) for i in range(num_qubits)])

def qaoa_layer(gamma, beta):
    qml.qaoa.cost_layer(gamma, H_ising)
    qml.qaoa.mixer_layer(beta, mixer_h)

@qml.qnode(dev)
def cost_function(params):
    # params is a 2D array of shape (2, p) where p is the number of QAOA layers
    gammas = params[0]
    betas = params[1]

    # 1. Initialize in a superposition state
    for i in range(num_qubits):
        qml.Hadamard(wires=i)

    # 2. Apply p layers of QAOA
    for i in range(len(gammas)):
        qaoa_layer(gammas[i], betas[i])

    # 3. Measure the expectation value of the cost Hamiltonian
    return qml.expval(H_ising)

# Define depth 'p'. Start small (p=2) on your M1 to gauge execution time before scaling.
p = 2
np.random.seed(42)
# Initialize parameters close to zero to avoid barren plateaus
initial_params = np.random.uniform(low=-0.01, high=0.01, size=(2, p), requires_grad=True)

In [8]:
# Re-declare device and QNode for optimization ensuring backprop
dev = qml.device('default.qubit', wires=range(num_qubits))

@qml.qnode(dev, diff_method="backprop")
def cost_function(params):
    # Hadamard Transform
    for i in range(num_qubits):
        qml.Hadamard(wires=i)

    # Applying the qaoa layers
    for i in range(len(params[0])):
        qaoa_layer(params[0][i], params[1][i])
    return qml.expval(H_ising)

# Optimization Execution
opt = qml.AdamOptimizer(stepsize=0.05)
epochs = 150
current_params = initial_params
lowest_param_set = (float('inf'), current_params)

print("Commencing QAOA Optimization...")
for epoch in range(epochs):
    current_params, cost = opt.step_and_cost(cost_function, current_params)
    if cost < lowest_param_set[0]:
        lowest_param_set = (cost, current_params)
    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | Cost: {cost:.4f}")

print("Optimization converged.")

Commencing VQE/QAOA Optimization...
Epoch   0 | Cost: 4876.4807
Epoch  10 | Cost: 3619.2827
Epoch  20 | Cost: 8032.0083
Epoch  30 | Cost: 5425.6738
Epoch  40 | Cost: 4972.8794
Epoch  50 | Cost: 5107.2784
Epoch  60 | Cost: 4923.3494
Epoch  70 | Cost: 5078.8280
Epoch  80 | Cost: 4799.2886
Epoch  90 | Cost: 4795.1011
Epoch 100 | Cost: 4890.7266
Epoch 110 | Cost: 4605.3720
Epoch 120 | Cost: 4828.7628
Epoch 130 | Cost: 4661.0039
Epoch 140 | Cost: 5176.1901
Optimization converged.


In [9]:
@qml.qnode(dev)
def sample_function(params):
    for i in range(num_qubits):
        qml.Hadamard(wires=i)
    for i in range(len(params[0])):
        qaoa_layer(params[0][i], params[1][i])
    return qml.probs(wires=range(num_qubits))

# Generate the probability vector for all 2^19 states
probabilities = sample_function(current_params)

In [14]:
top_k = 100
top_indices = np.argsort(probabilities)[-top_k:][::-1]
top_indices

tensor([ 83203,  75011,  83200,  75008,  70915,  68867,  70912,  68864,
        148739, 140547, 134403, 136451, 148736, 140544,  83210,  75018,
             0, 134400, 136448,  70922,  68874, 148746, 140554, 134410,
        136458,  36099,  38147,  50435,  42243,  36096,  38144,  36106,
         38154,  50432,  42240,  50442,  42250, 148883, 148899, 140691,
        140707, 134547, 136595, 134563, 136611, 148819, 148835, 140627,
        140643, 134483, 134499, 136531, 136547,  36243,  36259,  38291,
         38307,  36179,  36195,  38227,  38243,  83347,  83363,  75155,
         75171,  83283,  83299,  75091,  75107,  83395,  75203,  71059,
         69011,  71075,  69027, 148931, 140739,  70995,  68947,  71011,
         68963, 134595, 136643,  83206,  75014,  50579,  50595,  42387,
         42403,  71107,  69059,  50515,  50531,  42323,  42339,  36291,
         38339,  70918,  68870, 148742], requires_grad=True)

In [15]:

# 1. Extract the top N most probable bitstrings
top_k = 100
# np.argsort returns indices; we take the last 'top_k' and reverse them for descending order
top_indices = np.argsort(probabilities)[-top_k:][::-1]

valid_conformations = []

seq_positions = sorted(list(h_flex_linear.keys()))
wire_offsets = {}
current_wire = 0
for seq in seq_positions:
    wire_offsets[seq] = current_wire
    current_wire += len(h_flex_linear[seq])


# Helper to convert integer index back to binary list (length = 19)
def int_to_bitstring(idx, length):
    return [int(x) for x in format(idx, f'0{length}b')]

# 2. Enforce the One-Hot Constraint
for idx in top_indices:
    bitstring = int_to_bitstring(idx, num_qubits)
    print(bitstring)
    is_valid = True

    # Iterate through each residue's allocated wires using your existing `wire_offsets`
    # and the known length of h_flex[seq]
    for seq in seq_positions:
        start_wire = wire_offsets[seq]
        num_rots = len(h_flex_linear[seq])

        # Sum the bits corresponding to this residue's rotamers
        residue_sum = sum(bitstring[start_wire : start_wire + num_rots])

        if residue_sum != 1:
            is_valid = False
            break # Fails the penalty constraint

    if is_valid:
        # 3. Calculate True Biological Energy (Classical PyRosetta Equation)
        # using the valid bitstring against the original h_flex and J_flex tensors.
        bio_energy = 0 # calculate_classical_energy(bitstring, h_flex, J_flex, global_offset)
        valid_conformations.append({
            "bitstring": bitstring,
            "probability": probabilities[idx],
            "energy": bio_energy
        })

if not valid_conformations:
    raise ValueError("Zero valid conformations found in the top sampled states. You must increase QAOA depth 'p' or increase the penalty multiplier.")

# Sort the strictly valid conformations by their true biological energy
valid_conformations.sort(key=lambda x: x['energy'])
best_conformation = valid_conformations[0]

print(f"Optimal Valid Sequence: {best_conformation['bitstring']}")
print(f"Classical Energy: {best_conformation['energy']} kcal/mol")

ValueError: Zero valid conformations found in the top sampled states. You must increase QAOA depth 'p' or increase the penalty multiplier.